# Stage 2 MLP SHAP (GradientExplainer, within-cohort CV)

This notebook computes out-of-fold (OOF) SHAP values for the synthesis-winning Stage 2 **MLP** on long-format rows (animal × frequency × SPL level).

Cohort mapping:
- Cohort A (Liberman) → within-cohort scenario **C**
- Cohort B (Brad) → within-cohort scenario **A**

Outputs:
- Parquet: `figures/cache/shap_mlp/`
- Figures (PNG + SVG): `figures/shap_mlp/`


In [2]:
from __future__ import annotations

import numpy as np
import pandas as pd
from pathlib import Path

from utils.benchmark_metrics import apply_slide_rcparams
from utils.liberman_classical import (
    liberman_feature_lists,
    stage2_feature_lists,
)
from utils.nn_stage2_data import load_nn_stage2_data, splits_for_long_stage2
from utils.stage2_hp import resolve_torch_device
from utils.stage2_mlp_shap import (
    COHORT_SLUGS_DECK,
    compute_fold_shap,
    export_feature_names_mlp,
    aggregate_oof_long_grain,
    export_mlp_shap_parquet,
    fit_mlp_fold,
    fold_cache_path,
    load_fold_cache,
    aggregate_oof_animal_freq_grain,
    mean_abs_shap_tables_by_cohort,
    mlp_hp_for_scenario,
    run_mlp_shap_deck_figures,
    save_fold_cache,
)
from utils.stage2_synthesis_cv import (
    _mlp_holdout_seed,
    attach_global_stage1,
    build_cv_scenario_frames,
    fit_global_stage1_for_scenario,
    generate_cv_folds,
)
from utils.subject_cv import DEFAULT_CEILING_CV_RANDOM_STATE

apply_slide_rcparams()
device = resolve_torch_device()
print("device:", device)

SCENARIO_BY_COHORT = {"buran": "A", "liberman": "C"}
COHORT_TO_EVAL = {"buran": "Brad", "liberman": "Liberman"}

N_SPLITS = 10
FORCE_RERUN = False
MAX_FOLDS = None
MAX_BACKGROUND = None  # full training-fold background (per spec)

CACHE_DIR = Path("figures/cache/shap_mlp")
FIG_DIR = Path("figures/shap_mlp")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

data = load_nn_stage2_data()
splits = splits_for_long_stage2(data)

feats = liberman_feature_lists(data.reformatted_orig, data.common_cols)
_sn, _sl, _sc, long_num, long_cat = stage2_feature_lists(
    feats, noise_label="predicted"
)
long_log = list(feats["long_log"])

feature_names = long_num + list(long_cat) + long_log
export_feature_names_mlp(feature_names, cache_dir=CACHE_DIR)

global_s1 = {
    scen: fit_global_stage1_for_scenario(scen, data, splits, verbose=False)
    for scen in ("A", "B", "C")
}

cohort_outputs: dict[str, dict] = {}

for cohort_slug in COHORT_SLUGS_DECK:
    eval_cohort = COHORT_TO_EVAL[cohort_slug]
    scen = SCENARIO_BY_COHORT[cohort_slug]
    hp = mlp_hp_for_scenario(scen)

    wide_all = (
        data.reformatted.reset_index(drop=True)
        if eval_cohort == "Brad"
        else data.reformatted_orig.reset_index(drop=True)
    )
    folds = generate_cv_folds(
        wide_all,
        n_splits=N_SPLITS,
        random_state=DEFAULT_CEILING_CV_RANDOM_STATE,
    )
    n_folds = len(folds) if MAX_FOLDS is None else min(MAX_FOLDS, len(folds))

    shap_rows, feat_rows, meta_rows = [], [], []

    for fold_id in range(n_folds):
        _tr_idx, _te_idx, tr_anim, te_anim = folds[fold_id]

        cache_p = fold_cache_path(CACHE_DIR, cohort_slug, fold_id)
        if cache_p.is_file() and not FORCE_RERUN:
            shap_df, feat_df, meta_df = load_fold_cache(cache_p)
        else:
            w_tr, w_ev, l_tr, l_ev, _, _ = build_cv_scenario_frames(
                eval_cohort, scen, tr_anim, te_anim, data, splits
            )
            w_tr, w_ev, l_tr, l_ev = attach_global_stage1(
                w_tr, w_ev, l_tr, l_ev, global_s1[scen]["animal_preds"]
            )

            holdout_seed = _mlp_holdout_seed(eval_cohort, scen, fold_id)

            art = fit_mlp_fold(
                l_tr,
                l_ev,
                long_num,
                long_cat,
                long_log,
                hp,
                holdout_seed=holdout_seed,
                device=device,
            )

            shap_df, feat_df = compute_fold_shap(
                art,
                holdout_seed=holdout_seed,
                max_background=MAX_BACKGROUND,
            )

            meta_df = art.meta_eval.copy()
            meta_df["fold_id"] = fold_id
            meta_df["cohort"] = cohort_slug
            meta_df["rmse_animal_freq"] = art.rmse_animal_freq

            save_fold_cache(
                cache_p, shap_df=shap_df, feat_df=feat_df, meta_df=meta_df
            )

            print(
                f"  {cohort_slug} fold {fold_id}: rows={len(shap_df)} "
                f"rmse_af={art.rmse_animal_freq:.4f}"
            )

        shap_rows.append(shap_df)
        feat_rows.append(feat_df)
        meta_rows.append(meta_df)

    shap_oof = pd.concat(shap_rows, ignore_index=True)
    feat_oof = pd.concat(feat_rows, ignore_index=True)
    meta_oof = pd.concat(meta_rows, ignore_index=True)

    shap_oof, feat_oof, meta_oof = aggregate_oof_long_grain(
        shap_oof, feat_oof, meta_oof
    )
    cohort_outputs[cohort_slug] = {
        "shap": shap_oof,
        "feat": feat_oof,
        "meta": meta_oof,
    }
    export_mlp_shap_parquet(
        cohort_slug, shap_oof, feat_oof, meta_oof, cache_dir=CACHE_DIR
    )
    print("exported", cohort_slug, "rows", len(meta_oof))

device: mps
exported liberman rows 5901
exported buran rows 2781


In [3]:
COHORT_DISPLAY = {"liberman": "Cohort A (Liberman)", "buran": "Cohort B (Brad)"}

# Mean |SHAP| at animal × frequency (SPL levels averaged within each pair).
tables = mean_abs_shap_tables_by_cohort(cohort_outputs, grain="animal_freq")
for slug in COHORT_SLUGS_DECK:
    tbl = tables[slug]
    shap_af, _, _ = aggregate_oof_animal_freq_grain(
        cohort_outputs[slug]["shap"],
        cohort_outputs[slug]["feat"],
        cohort_outputs[slug]["meta"],
    )
    assert tbl["mean_abs_shap"].is_monotonic_decreasing
    assert tbl["rank"].iloc[0] == 1
    print(
        f"{COHORT_DISPLAY[slug]}  "
        f"(n_animal_freq_rows={len(shap_af):,}; mean |SHAP| pooled over SPL)"
    )
    display(
        tbl.assign(
            mean_abs_shap=lambda d: d["mean_abs_shap"].map("{:.5f}".format)
        )
    )

Cohort A (Liberman)  (n_animal_freq_rows=616; mean |SHAP| pooled over SPL)


,feature,paper_label,mean_abs_shap,rank
0,frequency,Frequency (kHz),0.86775,1
1,strain_binary,Strain (CBA/CaJ vs C57BL/6J),0.73615,2
2,noise_preds,Predicted noise,0.55357,3
3,total_variance,Total variance,0.39772,4
4,amplitude,Amplitude,0.39215,5
5,slope,Wave I slope,0.37793,6
6,PeakILateCurvature,Peak I curvature (late),0.35088,7
7,PeakIEarlyCurvature,Peak I curvature (early),0.34022,8
8,TroughIEarlyCurvature,Trough I curvature (early),0.33661,9
9,level,SPL (dB),0.33354,10


Cohort B (Brad)  (n_animal_freq_rows=360; mean |SHAP| pooled over SPL)


,feature,paper_label,mean_abs_shap,rank
0,frequency,Frequency (kHz),0.96681,1
1,total_variance,Total variance,0.56690,2
2,amplitude,Amplitude,0.54311,3
3,slope,Wave I slope,0.52345,4
4,distance,Peak-to-trough latency,0.51565,5
5,noise_preds,Predicted noise,0.46667,6
6,PeakILateCurvature,Peak I curvature (late),0.46058,7
7,TroughIEarlyCurvature,Trough I curvature (early),0.45795,8
8,TroughILateCurvature,Trough I curvature (late),0.43973,9
9,PeakIEarlyCurvature,Peak I curvature (early),0.41973,10


In [4]:
paths = run_mlp_shap_deck_figures(cohort_outputs, FIG_DIR)
print("wrote", len(paths), "figure files to", FIG_DIR)

wrote 10 figure files to figures/shap_mlp
